<a href="https://colab.research.google.com/github/NasorHidar/fly-rank-ml-1/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NasorHidar/fly-rank-ml-1/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split, GroupKFold

# Rule: Stay reproducible
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Load the dataset
url = "https://raw.githubusercontent.com/NasorHidar/fly-rank-ml-1/refs/heads/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Recreate Week 5 Target and drop leaky columns
df['needs_refresh'] = (df['trend_pct'] < 0).astype(int)
df_clean = df.drop(columns=['content_id', 'trend_pct', 'trend_direction'])

print(f"Data loaded. Shape: {df_clean.shape}")

Data loaded. Shape: (30000, 42)


## 1. Two paper findings + my methodology questions

**Finding 1:** *"AI-generated content experiences a faster decay in organic traffic over 6 months compared to human-written content."*
*   **Methodology Question:** How is the "AI-generated" label defined and sourced? If it relies on third-party AI detectors, does the validation design account for the known false-positive rates of those tools? To support this claim, we would need to know the label comes from a controlled environment where the provenance of the content is definitively known.

**Finding 2:** *"Implementing a content refresh strategy guarantees a reversal of negative traffic trends within 30 days."*
*   **Methodology Question:** Where does the evidence for a "reversal" come from, and is it isolated from external factors? The word "guarantees" implies a strong causal link. Does the validation design utilize a randomized control group (e.g., A/B testing similar declining pages where half are left untouched) to prove the refresh caused the reversal, rather than seasonal shifts or algorithm updates?

## 2. My model under an honest split (before/after)

Split Analysis:
The "Before" metric uses a naive random split. In agency data, this causes leakage because the model learns client-specific behaviors in the training set and simply regurgitates them in the validation set.

The "After" metric uses GroupKFold grouped by client_id. This is an honest split because it simulates the reality of onboarding a brand new client. The measured drop in AUC reflects the model's true capability to generalize to unseen clients, providing a safe, directional expectation for production performance.

In [2]:
# Separate features and target
X = df_clean.drop(columns=['needs_refresh'])
y = df_clean['needs_refresh']

# We need to isolate the 'client_id' for our honest split, then drop it from features
groups = X['client_id']
X = X.drop(columns=['client_id'])

# Convert categorical columns to numeric
X = pd.get_dummies(X, drop_first=True)

model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)

# ---------------------------------------------------------
# BEFORE: Naive Random Split (High risk of data leakage across clients)
# ---------------------------------------------------------
X_train_rand, X_val_rand, y_train_rand, y_val_rand = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)
model.fit(X_train_rand, y_train_rand)
y_pred_rand = model.predict_proba(X_val_rand)[:, 1]
naive_auc = roc_auc_score(y_val_rand, y_pred_rand)

# ---------------------------------------------------------
# AFTER: Honest Grouped Split (Grouped by Client)
# ---------------------------------------------------------
# This ensures a client's data is either entirely in the train set OR the validation set, never both.
gkf = GroupKFold(n_splits=5)
train_idx, val_idx = next(gkf.split(X, y, groups=groups))

X_train_grp, X_val_grp = X.iloc[train_idx], X.iloc[val_idx]
y_train_grp, y_val_grp = y.iloc[train_idx], y.iloc[val_idx]

model.fit(X_train_grp, y_train_grp)
y_pred_grp = model.predict_proba(X_val_grp)[:, 1]
honest_auc = roc_auc_score(y_val_grp, y_pred_grp)

print("--- Split Design Comparison ---")
print(f"Naive Random Split AUC:      {naive_auc:.4f}")
print(f"Honest Grouped Split AUC:    {honest_auc:.4f}")
print(f"Difference (Reality Check):  {honest_auc - naive_auc:.4f}")

--- Split Design Comparison ---
Naive Random Split AUC:      0.9603
Honest Grouped Split AUC:    0.8920
Difference (Reality Check):  -0.0683


## 3. Leakage audit

Leakage Audit Results:
I reviewed the top correlated features against the target. There are no suspiciously high correlations (e.g., > 0.85) that would indicate target leakage. Features like trend_pct and trend_direction were properly excluded prior to splitting. The remaining features (impressions_90d, clicks_90d, etc.) represent purely historical observations, confirming that the model is predicting based on past data without cheating using future metrics.

In [3]:
# Calculate absolute Pearson correlation between numeric features and the target
# We are hunting for anything suspiciously close to 1.0 or -1.0
numeric_df = X.copy()
numeric_df['TARGET_needs_refresh'] = y

correlations = numeric_df.corr()['TARGET_needs_refresh'].abs().sort_values(ascending=False)

print("--- Top 10 Feature Correlations with Target ---")
print(correlations.head(11)[1:]) # Skip the target's 1.0 correlation with itself

--- Top 10 Feature Correlations with Target ---
days_with_impressions                0.343823
impression_tier_low                  0.261668
position_tier_top_3                  0.235325
content_type_feedly article          0.198738
content_type_keyword article         0.169532
word_count_tier_<1000                0.167064
char_count_tier_<8000                0.164182
word_count                           0.151164
model_used_gemini-3-flash-preview    0.146873
freshness_tier_91-180                0.137971
Name: TARGET_needs_refresh, dtype: float64


## 4. Claim rewrite

**Original Bold Claim:**
"High impressions combined with a low CTR causes a page's organic traffic to drop, so running our refresh pipeline on these pages will fix their performance."

**Rewritten in Safe Language:**
"We observed a strong directional correlation between historical segments with high impressions and low CTRs, and subsequent negative traffic trends. Measuring this signal provides decision-support to help prioritize which pages the team should audit for potential content refreshes."

### Self-check

Before you submit, confirm each line honestly:

*   [x] Every section above is filled — markdown thinking AND the code that backs it
*   [x] The notebook runs top to bottom with no errors (Runtime → Run all)
*   [x] No client names, URLs, or private queries anywhere
*   [x] My claims use careful words: observed, measured, directional, decision-support
*   [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.